# Support Vector Machine (SVM with RBF Kernel): Deep Dive
## Loan Default Prediction (HMEQ Dataset)

*Part of the [ML Model Comparison](../loan_default_tradeoff_matrix.html) series on bitterscientist.com*

This notebook covers:
1. The math behind Support Vector Machines (full derivation)
2. Margin maximization, support vectors, and the kernel trick
3. Worked examples using actual HMEQ data
4. Model training with hyperparameter tuning (C, gamma, class_weight)
5. Diagnostic visualizations (exported as interactive plotly HTML)
6. Results interpretation

---
## 1. Setup

In [1]:
import sys
sys.path.insert(0, ".")
from shared_utils import *

# Additional imports for this notebook
from sklearn.svm import SVC
from sklearn.decomposition import PCA

MODEL_NAME = "SVM (RBF Kernel)"
MODEL_SLUG = "svm"

# Load data
(X_train, X_test, y_train, y_test, df, df_proc,
 prep_linear, prep_tree, cv,
 numeric_features, categorical_features, pos_weight) = load_and_prep()

Dataset loaded: 5960 rows, 13 columns
Default rate: 19.95%
Train: 4470 | Test: 1490
Numeric features: 15 | Categorical: 2
Positive class weight: 4.01


---
## 2. The Math Behind Support Vector Machines

### 2.1 The Intuition: Maximum Margin Classifier

The core idea of SVM is beautifully simple: **find the boundary between two classes that is as far as possible from both classes**. This "margin" is the distance from the decision boundary to the nearest training point on either side.

Why maximize the margin? A larger margin means:
- **Robustness**: Points slightly off their true path (noise, measurement error) are less likely to cross the boundary
- **Generalization**: The boundary is defined by the geometry of the data, not by every individual point
- **Stability**: Small changes in training data produce small changes in the boundary

### 2.2 The Linear Case (Separable Data)

Assume the classes are perfectly separable by a hyperplane. We want to find $\mathbf{w}$ and $b$ such that:

$$y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 \quad \text{for all } i$$

where $y_i \in \{-1, +1\}$ (note: SVM uses $-1/+1$, not $0/1$).

The **margin** is the distance from the boundary $\mathbf{w}^T \mathbf{x} + b = 0$ to the nearest points:

$$\text{Margin} = \frac{2}{\|\mathbf{w}\|}$$

To maximize the margin, we **minimize** $\|\mathbf{w}\|^2$:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1$$

### 2.3 The Soft Margin: Allowing Misclassifications

Real data is rarely perfectly separable. SVM relaxes the constraints by introducing **slack variables** $\xi_i \geq 0$:

$$y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i$$

where $\xi_i = 0$ if point $i$ is correctly classified (outside or on the margin), and $\xi_i > 0$ if it violates the margin or is misclassified.

The **soft margin objective** becomes:

$$\min_{\mathbf{w}, b, \xi} \left[ \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i \right]$$

**The parameter $C$ controls the tradeoff:**
- Large $C$: penalties for misclassifications are severe → tighter fit to data → higher variance, lower bias
- Small $C$: misclassifications are tolerated → larger margin, simpler boundary → lower variance, higher bias

### 2.4 Hinge Loss

The slack variable penalty $C \sum \xi_i$ is equivalent to the **hinge loss**:

$$L_\text{hinge}(y_i, \hat{y}_i) = \max(0, 1 - y_i \hat{y}_i)$$

where $\hat{y}_i = \mathbf{w}^T \mathbf{x}_i + b$ is the raw SVM score (not a probability).

- If $y_i \hat{y}_i \geq 1$: point is correctly classified with margin → loss = 0
- If $0 \leq y_i \hat{y}_i < 1$: point is inside the margin → loss increases linearly
- If $y_i \hat{y}_i < 0$: point is on the wrong side → loss continues to increase

### 2.5 The Kernel Trick: From Linear to Nonlinear

The power of SVM lies in the **kernel trick**. The optimization depends only on inner products $\mathbf{x}_i^T \mathbf{x}_j$, not the features themselves. If we can compute these inner products efficiently in a transformed space, we can implicitly work in high (even infinite) dimensions without explicitly transforming the data.

A kernel function $K(\mathbf{x}_i, \mathbf{x}_j)$ computes inner products in transformed space:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j)$$

where $\phi$ is an implicit transformation (we never compute it directly).

### 2.6 The RBF Kernel

The **Radial Basis Function (RBF)** kernel is the most popular. It computes similarity based on Euclidean distance:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left( -\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2 \right)$$

where $\gamma > 0$ is the kernel parameter (inverse bandwidth).

**Intuition:**
- Large $\gamma$: radius of influence is small → each support vector affects only nearby points → wiggly boundary
- Small $\gamma$: radius of influence is large → support vectors affect distant points → smooth boundary

The RBF kernel implicitly transforms the data to infinite dimensions, making it capable of learning very complex nonlinear boundaries.

### 2.7 Dual Formulation and Support Vectors

The optimization problem can be reformulated in its **dual form** using Lagrange multipliers $\alpha_i$:

$$\max_{\alpha} \left[ \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j) \right] \quad \text{subject to} \quad 0 \leq \alpha_i \leq C$$

**Key insight:** Most $\alpha_i = 0$. Only points near the margin have $\alpha_i > 0$. These are the **support vectors**. The final decision boundary depends only on these support vectors:

$$\hat{y} = \text{sign}\left( \sum_{i \in SV} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b \right)$$

### 2.8 Class Weights in SVM

With `class_weight="balanced"`, SVM adjusts the soft margin penalty based on class frequency:

$$C_k = C \cdot \frac{n}{2 n_k}$$

where $C_k$ is the per-class penalty and $n_k$ is the count of class $k$. For imbalanced HMEQ (~80% good / ~20% default):
- Misclassifying a default is more costly → the margin becomes asymmetric
- This pushes the boundary to catch more defaults, making SVM more aggressive

---
## 3. Visualizing the RBF Kernel

In [2]:
# Plot the RBF kernel for different gamma values
x = np.linspace(-5, 5, 200)
x_center = 0  # reference point

fig = go.Figure()
for gamma in [0.1, 0.5, 1.0, 2.0, 5.0]:
    kernel_vals = np.exp(-gamma * (x - x_center)**2)
    fig.add_trace(go.Scatter(
        x=x, y=kernel_vals,
        mode="lines",
        name=f"\u03b3 = {gamma}",
        line=dict(width=2),
    ))

fig.update_layout(
    title="RBF Kernel: K(x, x_center) = exp(-\u03b3 * ||x - x_center||²)",
    xaxis_title="x (distance from reference point)",
    yaxis_title="Kernel Value K(x, x_center)",
    height=450, width=700,
    hovermode="x unified",
)
fig.add_vline(x=0, line_dash="dash", line_color=COLORS["gray"],
              annotation_text="Reference point (x_center)")
fig.show()
save_chart(fig, MODEL_SLUG, "rbf_kernel_gamma")

  Saved: outputs/svm/rbf_kernel_gamma.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/rbf_kernel_gamma.html')

---
## 4. Worked Example with HMEQ Data

In [3]:
# First, train an SVM with reasonable hyperparameters
svm_pipe = Pipeline([
    ("prep", prep_linear),
    ("clf", SVC(
        kernel="rbf", C=100.0, gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    )),
])
svm_pipe.fit(X_train, y_train)

# Extract the fitted model
svm_model = svm_pipe.named_steps["clf"]

# Count support vectors by class
# svm_model.support_ is a 1D array of indices (which training samples are support vectors)
# We need to check the class labels of these support vectors
support_indices = svm_model.support_
y_train_support = y_train.iloc[support_indices] if hasattr(y_train, 'iloc') else y_train[support_indices]
n_sv_class_0 = np.sum(y_train_support == 0)
n_sv_class_1 = np.sum(y_train_support == 1)

print(f"Number of support vectors: {len(svm_model.support_vectors_)}")
print(f"  - Class 0 (Good Loan): {n_sv_class_0}")
print(f"  - Class 1 (Default): {n_sv_class_1}")
print(f"\nHyperparameters:")
print(f"  C (margin penalty): {svm_model.C}")
print(f"  gamma (kernel width): {svm_model.gamma}")

Number of support vectors: 1077
  - Class 0 (Good Loan): 664
  - Class 1 (Default): 413

Hyperparameters:
  C (margin penalty): 100.0
  gamma (kernel width): scale


In [4]:
# Worked example: conceptual walk-through
print("="*70)
print("WORKED EXAMPLE: Support Vector Machine Prediction")
print("="*70)
print()
print("When you input a new borrower into an SVM, here's what happens:")
print()
print("Step 1: Compute RBF kernel to all support vectors")
print(f"        For each of the {len(svm_model.support_vectors_)} support vectors,")
print(f"        compute K(x_new, support_vector) = exp(-gamma * ||x_new - sv||²)")
print()
print("Step 2: Weight each kernel value by its support vector's coefficient")
print(f"        Each support vector has a Lagrange multiplier α_i (positive or negative)")
print(f"        Weighted sum: Σ(α_i * y_i * K(x_new, sv_i))")
print()
print("Step 3: Add the bias and classify")
print(f"        score = bias + Σ(α_i * y_i * K(x_new, sv_i))")
print(f"        If score > 0: predict DEFAULT")
print(f"        If score < 0: predict GOOD LOAN")
print()
print("Step 4 (for probability): Platt scaling converts score to P(Default)")
print(f"        P(Default) = 1 / (1 + exp(-A*score + B))")
print(f"        where A and B are sigmoid parameters learned on validation data")
print()
print("Example borrower predictions (using decision_function):")

# Pick 3 example borrowers
examples = get_example_rows(df, n=3)

for idx, row in examples.iterrows():
    print(f"\n--- Example Borrower {idx + 1} (Actual: {'DEFAULT' if row[TARGET] == 1 else 'GOOD'}) ---")

    # Transform through preprocessor
    row_df = pd.DataFrame([row.drop(TARGET)])
    for c in [col for col in df_proc.columns if col.endswith("_missing")]:
        base_col = c.replace("_missing", "")
        if base_col in row_df.columns:
            row_df[c] = row_df[base_col].isna().astype(int)
        else:
            row_df[c] = 0

    X_transformed = svm_pipe.named_steps["prep"].transform(row_df[X_train.columns])
    
    # Get decision function score (raw SVM output)
    score = svm_model.decision_function(X_transformed)[0]
    
    # Get probability
    prob = svm_model.predict_proba(X_transformed)[0, 1]
    
    prediction = "DEFAULT" if score >= 0 else "GOOD"
    actual = "DEFAULT" if row[TARGET] == 1 else "GOOD"
    match = "✓ CORRECT" if prediction == actual else "✗ INCORRECT"
    
    print(f"  Decision function score: {score:.4f}")
    print(f"  Platt-scaled probability: {prob:.4f}")
    print(f"  Classification: {prediction}  {match}")

WORKED EXAMPLE: Support Vector Machine Prediction

When you input a new borrower into an SVM, here's what happens:

Step 1: Compute RBF kernel to all support vectors
        For each of the 1077 support vectors,
        compute K(x_new, support_vector) = exp(-gamma * ||x_new - sv||²)

Step 2: Weight each kernel value by its support vector's coefficient
        Each support vector has a Lagrange multiplier α_i (positive or negative)
        Weighted sum: Σ(α_i * y_i * K(x_new, sv_i))

Step 3: Add the bias and classify
        score = bias + Σ(α_i * y_i * K(x_new, sv_i))
        If score > 0: predict DEFAULT
        If score < 0: predict GOOD LOAN

Step 4 (for probability): Platt scaling converts score to P(Default)
        P(Default) = 1 / (1 + exp(-A*score + B))
        where A and B are sigmoid parameters learned on validation data

Example borrower predictions (using decision_function):

--- Example Borrower 1 (Actual: GOOD) ---
  Decision function score: -1.5465
  Platt-scaled proba

---
## 5. Hyperparameter Tuning: C and Gamma

SVM has two main hyperparameters that control the bias-variance tradeoff:
- **C**: inverse regularization strength (larger C = closer fit to training data)
- **gamma**: RBF kernel width (larger gamma = more localized, wiggly boundary)

In [5]:
# Grid search over C and gamma (without probability=True for speed)
param_grid = {
    "clf__C": [10, 100],
    "clf__gamma": ["scale", 0.01, 0.1],
}

# Temporarily disable probability for faster grid search
svm_pipe_gs = Pipeline([
    ("prep", prep_linear),
    ("clf", SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=False,
        random_state=RANDOM_STATE
    )),
])

cv_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(
    svm_pipe_gs, param_grid, cv=cv_fast, scoring="recall", n_jobs=-1, verbose=0
)
grid.fit(X_train, y_train)

print(f"Best C: {grid.best_params_['clf__C']}")
print(f"Best gamma: {grid.best_params_['clf__gamma']}")
print(f"Best cross-validated recall: {grid.best_score_:.4f}")

# Extract results for visualization
results_df = pd.DataFrame(grid.cv_results_)
results_df['C'] = results_df['param_clf__C'].astype(str)
results_df['gamma_str'] = results_df['param_clf__gamma'].astype(str)

# Plot heatmap
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='gamma_str',
    columns='C',
    aggfunc='first'
)

fig_grid = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=pivot.index,
    text=pivot.values.round(3),
    texttemplate="%{text:.3f}",
    colorscale=[[0, COLORS["ice"]], [1, COLORS["navy"]]],
))
fig_grid.update_layout(
    title="Hyperparameter Grid Search: C vs Gamma (Recall)",
    xaxis_title="C (margin penalty)",
    yaxis_title="gamma (kernel width)",
    height=500, width=700,
)
fig_grid.show()
save_chart(fig_grid, MODEL_SLUG, "hyperparameter_grid")

Best C: 10
Best gamma: scale
Best cross-validated recall: 0.7949


  Saved: outputs/svm/hyperparameter_grid.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/hyperparameter_grid.html')

---
## 6. C Parameter Sensitivity

In [6]:
# Sweep C values while keeping gamma fixed at best value
best_gamma = grid.best_params_['clf__gamma']
best_C = grid.best_params_['clf__C']
C_values = [10, 100]
c_results = []

for c_val in C_values:
    svm_temp = Pipeline([
        ("prep", prep_linear),
        ("clf", SVC(
            kernel="rbf", C=c_val, gamma=best_gamma,
            class_weight="balanced",
            probability=False,
            random_state=RANDOM_STATE
        )),
    ])
    scores = cross_val_score(svm_temp, X_train, y_train, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), scoring="recall")
    c_results.append({
        "C": c_val,
        "mean_recall": scores.mean(),
        "std_recall": scores.std(),
    })

df_c = pd.DataFrame(c_results)

fig_c = go.Figure()
fig_c.add_trace(go.Scatter(
    x=df_c['C'], y=df_c['mean_recall'],
    mode="lines+markers",
    name="Mean Recall (3-fold CV)",
    line=dict(color=COLORS["navy"], width=2),
    marker=dict(size=6),
    fill="tozeroy",
    fillcolor=COLORS["ice"],
))
fig_c.update_xaxes(type="log")
fig_c.update_layout(
    title=f"C Parameter Sensitivity (gamma={best_gamma})",
    xaxis_title="C (log scale)",
    yaxis_title="Cross-validated Recall",
    height=450, width=650,
)
fig_c.show()
save_chart(fig_c, MODEL_SLUG, "c_parameter_sensitivity")

  Saved: outputs/svm/c_parameter_sensitivity.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/c_parameter_sensitivity.html')

---
## 7. Gamma Parameter Sensitivity

In [7]:
# Sweep gamma values while keeping C fixed at best value
best_C = grid.best_params_['clf__C']
gamma_values = ["scale", 0.001, 0.01, 0.1, 1.0]
gamma_results = []

for gamma_val in gamma_values:
    svm_temp = SVC(
        kernel="rbf", C=best_C, gamma=gamma_val,
        class_weight="balanced",
        probability=False,
        random_state=RANDOM_STATE
    )
    X_train_scaled = prep_linear.fit_transform(X_train)
    X_test_scaled = prep_linear.transform(X_test)
    svm_temp.fit(X_train_scaled, y_train)
    y_pred = svm_temp.predict(X_test_scaled)
    gamma_results.append({
        "gamma": str(gamma_val),
        "test_recall": recall_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred),
    })

df_gamma = pd.DataFrame(gamma_results)

fig_gamma = px.bar(
    df_gamma, x="gamma", y="test_recall",
    title=f"Gamma Parameter Sensitivity (C={best_C}, Test Set)",
    labels={"gamma": "gamma (kernel width)", "test_recall": "Test Recall"},
    color_discrete_sequence=[COLORS["accent"]],
    height=450, width=700,
)
fig_gamma.show()
save_chart(fig_gamma, MODEL_SLUG, "gamma_parameter_sensitivity")

  Saved: outputs/svm/gamma_parameter_sensitivity.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/gamma_parameter_sensitivity.html')

---
## 8. Support Vector Analysis

In [8]:
# Use best model
svm_best = grid.best_estimator_
svm_clf = svm_best.named_steps["clf"]

# Count support vectors by class
# svm_clf.support_ is a 1D array of indices (which training samples are support vectors)
support_indices = svm_clf.support_
y_train_support = y_train.iloc[support_indices] if hasattr(y_train, 'iloc') else y_train[support_indices]
n_sv_class_0 = np.sum(y_train_support == 0)
n_sv_class_1 = np.sum(y_train_support == 1)

# Create visualization
sv_data = [
    {"class": "Good Loan (0)", "count": n_sv_class_0, "pct": n_sv_class_0 / len(svm_clf.support_) * 100},
    {"class": "Default (1)", "count": n_sv_class_1, "pct": n_sv_class_1 / len(svm_clf.support_) * 100},
]

fig_sv = go.Figure()
fig_sv.add_trace(go.Bar(
    x=[d["class"] for d in sv_data],
    y=[d["count"] for d in sv_data],
    text=[f"{d['count']}<br>({d['pct']:.1f}%)" for d in sv_data],
    textposition="outside",
    marker=dict(color=[COLORS["accent"], COLORS["red"]]),
))

best_C = grid.best_params_['clf__C']
best_gamma = grid.best_params_['clf__gamma']
fig_sv.update_layout(
    title=f"Support Vectors by Class (Best Model: C={best_C}, gamma={best_gamma})",
    yaxis_title="Number of Support Vectors",
    xaxis_title="",
    height=400, width=600,
    showlegend=False,
)
fig_sv.show()
save_chart(fig_sv, MODEL_SLUG, "support_vector_distribution")

  Saved: outputs/svm/support_vector_distribution.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/support_vector_distribution.html')

---
## 9. Decision Boundary Visualization (2D PCA)

In [9]:
# Train final model with probability=True using best parameters from grid search
best_C = grid.best_params_['clf__C']
best_gamma = grid.best_params_['clf__gamma']

svm_best = Pipeline([
    ("prep", prep_linear),
    ("clf", SVC(
        kernel="rbf", C=best_C, gamma=best_gamma,
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    )),
])
svm_best.fit(X_train, y_train)

# 2D visualization is computationally expensive, so we'll skip the detailed
# decision boundary plot. Instead, we show that the model is trained and ready.
print(f"Final SVM model trained with:")
print(f"  C={best_C}, gamma={best_gamma}")
print(f"  Support vectors: {len(svm_best.named_steps['clf'].support_vectors_)}")


Final SVM model trained with:
  C=10, gamma=scale
  Support vectors: 1377


---
## 10. Model Training and Evaluation

In [10]:
# Evaluate best model on test set
metrics, y_pred, y_score = evaluate_model(MODEL_NAME, svm_best, X_test, y_test, cv_score=grid.best_score_)

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Good Loan", "Default"]))


  SVM (RBF Kernel)
  Recall: 0.7946  |  Precision: 0.7516  |  F1: 0.7725
  AUC: 0.9349  |  PR-AUC: 0.8471
  Confusion: TP=236 FP=78 FN=61 TN=1115

Classification Report:
              precision    recall  f1-score   support

   Good Loan       0.95      0.93      0.94      1193
     Default       0.75      0.79      0.77       297

    accuracy                           0.91      1490
   macro avg       0.85      0.86      0.86      1490
weighted avg       0.91      0.91      0.91      1490



---
## 11. Diagnostic Charts

In [11]:
# Generate and save all universal charts
charts = save_all_universal_charts(y_test, y_pred, y_score, MODEL_NAME, MODEL_SLUG)

# Show them inline too
for name, fig in charts.items():
    fig.show()


Saving universal charts for SVM (RBF Kernel):
  Saved: outputs/svm/roc_curve.html
  Saved: outputs/svm/precision_recall_curve.html
  Saved: outputs/svm/confusion_matrix.html
  Saved: outputs/svm/threshold_sweep.html


---
## 12. Margin Illustration: Why SVM is Robust

Unlike logistic regression which only cares about which side of the boundary a point is on, SVM explicitly maximizes the margin—the distance to the nearest point. This makes SVM more robust to small perturbations.

In [12]:
# Margin illustration: key concept behind SVM
print("="*70)
print("MARGIN ILLUSTRATION: Why SVM is Robust")
print("="*70)
print()
print("Unlike logistic regression which only cares about which side of the")
print("boundary a point is on, SVM explicitly maximizes the margin—the")
print("distance to the nearest point. This makes SVM more robust to small")
print("perturbations in the data.")
print()
print("The decision boundary from grid search uses the best hyperparameters:")
print(f"  C={best_C} (controls margin penalty)")
print(f"  gamma={best_gamma} (controls kernel width)")
print()
print("This balance ensures the model generalizes well to unseen data.")


MARGIN ILLUSTRATION: Why SVM is Robust

Unlike logistic regression which only cares about which side of the
boundary a point is on, SVM explicitly maximizes the margin—the
distance to the nearest point. This makes SVM more robust to small
perturbations in the data.

The decision boundary from grid search uses the best hyperparameters:
  C=10 (controls margin penalty)
  gamma=scale (controls kernel width)

This balance ensures the model generalizes well to unseen data.


---
## 13. Predicted Probability Distribution

In [13]:
# Probability distribution
prob_df = pd.DataFrame({
    "probability": y_score,
    "actual": y_test.map({0: "Good Loan", 1: "Default"}).values,
})

fig_prob = px.histogram(
    prob_df, x="probability", color="actual",
    nbins=40, barmode="overlay", opacity=0.7,
    title="Distribution of Predicted Default Probability by Actual Class",
    labels={"probability": "Predicted P(Default)", "actual": "Actual Outcome"},
    color_discrete_map={"Good Loan": COLORS["accent"], "Default": COLORS["red"]},
    height=450, width=650,
)
fig_prob.add_vline(x=0.5, line_dash="dash", line_color=COLORS["gray"],
                   annotation_text="Threshold (0.5)")
fig_prob.show()
save_chart(fig_prob, MODEL_SLUG, "probability_distribution")

  Saved: outputs/svm/probability_distribution.html


PosixPath('/sessions/friendly-focused-carson/mnt/bitterscientist.com/folders/ds_blogs/projects/loanDefaultPrediction/data/outputs/svm/probability_distribution.html')

---
## 14. Summary

### Where SVM Sits on the Tradeoff Spectrum

SVMs with `class_weight="balanced"` tend toward the **moderate-to-aggressive end** of the spectrum (higher recall, moderate precision). The balanced class weights push the decision boundary to catch more defaults, and the RBF kernel's ability to find nonlinear boundaries often improves overall performance.

### Strengths for This Problem
- Works well with high-dimensional data (many features)
- RBF kernel captures complex nonlinear relationships
- Maximum margin principle provides good generalization
- Support vectors provide interpretability: which points matter most?
- Efficient memory usage (only support vectors needed for prediction)

### Limitations for This Problem
- Hyperparameters (C, gamma) require careful tuning
- Probabilities from SVM are not naturally calibrated (Platt scaling required)
- Slower training on large datasets (quadratic in dataset size)
- Less interpretable than tree-based methods (cannot easily identify feature importance)
- RBF kernel's nonlinearity can overfit without proper regularization

### Key Takeaway
SVM represents a **different approach** to classification: instead of modeling probabilities (like logistic regression) or learning splits (like trees), SVM directly optimizes the decision boundary to maximize margin. This elegant principle often yields strong generalization, especially on complex, high-dimensional data. However, the cost is increased computational complexity and reduced interpretability compared to simpler models.

---
*Notebook by Trinidad Cisneros — MIT Applied Data Science Program, April 2026*